# Specimen Count Checker
Use the inputs below to specify which schema and table to query for specimen counts.

In [ ]:
# --- Widget Inputs ---
# In Databricks, these create interactive text boxes at the top of the notebook.
# Locally (Jupyter), they fall back to default values defined below.

try:
    dbutils.widgets.text("schema", "", "Schema Name")
    dbutils.widgets.text("table", "", "Table Name")
    dbutils.widgets.text("specimen_col", "specimen_id", "Specimen ID Column")

    schema = dbutils.widgets.get("schema")
    table = dbutils.widgets.get("table")
    specimen_col = dbutils.widgets.get("specimen_col")
except NameError:
    # Running locally — set defaults here
    schema = "my_schema"
    table = "my_table"
    specimen_col = "specimen_id"

if not schema or not table:
    raise ValueError("Please provide both a schema and table name.")

full_table = f"`{schema}`.`{table}`"
print(f"Target table : {full_table}")
print(f"Specimen col : {specimen_col}")

## 1. Total Row Count

In [ ]:
total_count = spark.sql(f"SELECT COUNT(*) AS total_rows FROM {full_table}").collect()[0]["total_rows"]
print(f"Total rows in {full_table}: {total_count:,}")

## 2. Distinct Specimen Count

In [ ]:
distinct_count = spark.sql(
    f"SELECT COUNT(DISTINCT {specimen_col}) AS distinct_specimens FROM {full_table}"
).collect()[0]["distinct_specimens"]

print(f"Distinct specimens ({specimen_col}): {distinct_count:,}")

## 3. Null / Missing Specimen IDs

In [ ]:
null_count = spark.sql(
    f"SELECT COUNT(*) AS null_specimens FROM {full_table} WHERE {specimen_col} IS NULL"
).collect()[0]["null_specimens"]

print(f"Rows with NULL {specimen_col}: {null_count:,}")

## 4. Duplicate Specimen IDs

In [ ]:
duplicates_df = spark.sql(f"""
    SELECT {specimen_col}, COUNT(*) AS occurrence_count
    FROM {full_table}
    WHERE {specimen_col} IS NOT NULL
    GROUP BY {specimen_col}
    HAVING COUNT(*) > 1
    ORDER BY occurrence_count DESC
""")

dup_count = duplicates_df.count()
print(f"Specimen IDs with duplicates: {dup_count:,}")

if dup_count > 0:
    print("\nTop duplicates:")
    duplicates_df.show(20, truncate=False)

## 5. Summary Table

In [ ]:
summary = {
    "Table": full_table,
    "Total Rows": f"{total_count:,}",
    "Distinct Specimens": f"{distinct_count:,}",
    "Null Specimen IDs": f"{null_count:,}",
    "Specimen IDs w/ Duplicates": f"{dup_count:,}",
}

print("=" * 45)
print("SPECIMEN COUNT SUMMARY")
print("=" * 45)
for k, v in summary.items():
    print(f"{k:<30} {v}")
print("=" * 45)